## Collect and Consolidate Raw Datasets
### Description
Gather data from Kaggle, Hotel booking demand . Ensure that all datasets are properly stored, versioned, and documented for later cleaning and transformation.

### Download dataset from Kaggle

In [3]:
# -----------------------------
# Import libraries
# -----------------------------
import kagglehub
import pandas as pd
import os

# -----------------------------
# Define paths
# -----------------------------
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))
RAW_DATA_DIR = os.path.join(BASE_DIR, "data", "raw")

os.makedirs(RAW_DATA_DIR, exist_ok=True)
print("Base directory:", BASE_DIR)
print("Raw data directory:", RAW_DATA_DIR)


# -----------------------------
# Download dataset from Kaggle
# -----------------------------
print("Downloading dataset from Kaggle...")
path = kagglehub.dataset_download("jessemostipak/hotel-booking-demand")

file_src = os.path.join(path, "hotel_bookings.csv")
file_dst = os.path.join(RAW_DATA_DIR, "hotel_bookings.csv")

# Copy to /data/raw for version control
if not os.path.exists(file_dst):
    pd.read_csv(file_src).to_csv(file_dst, index=False)
    print(f" Dataset saved to: {file_dst}")
else:
    print(f" Using existing dataset at: {file_dst}")


Base directory: /Users/mati/hotel_pricing_project
Raw data directory: /Users/mati/hotel_pricing_project/data/raw
 Using existing dataset at: /Users/mati/hotel_pricing_project/data/raw/hotel_bookings.csv


### Data Cleansing

#### Hotel & Booking Dataset Description 
- **hotel**: Type of hotel – either Resort Hotel or City Hotel (we will not use this)
- **is_canceled**: Indicates whether the booking was canceled (1 = yes, 0 = no)
- **lead_time**: Number of days between the booking date and the arrival date
- **arrival_date_year**: Year of the guest's arrival
- **arrival_date_month**: Month of the guest's arrival with 12 categories: "January" to "December"
- **arrival_date_week_number**: Week number of the arrival date
- **arrival_date_day_of_month**: Day of the month of the arrival date

#### Stay Details

- **stays_in_weekend_nights**: Number of weekend nights (Saturday or Sunday) spent at the hotel
- **stays_in_week_nights**: Number of week nights (Monday to Friday) spent at the hotel
- **adults**: Number of adults in the booking
- **children**: Number of children in the booking
- **babies**: Number of babies in the booking

#### Meal & Market Information

- **meal**: Type of meal booked: BB = Bed & Breakfast, HB = Half Board, FB = Full Board, SC = Self Catering
- **country**: Country of origin of the guest (ISO country code, e.g., PRT = Portugal)
- **market_segment**: Market segment that generated the booking (Online TA, Offline TA/TO, Direct, Corporate, etc.)
- **distribution_channel**: Booking distribution channel (Direct, TA/TO, Corporate, GDS, etc.)

#### Guest Behavior

- **is_repeated_guest**: Indicates whether the guest has stayed at the hotel before (1 = yes, 0 = no)
- **previous_cancellations**: Number of previous bookings by this customer that were canceled
- **previous_bookings_not_canceled**: Number of previous bookings that were not canceled

#### Room & Booking Changes

- **reserved_room_type**: Type of room originally reserved
- **assigned_room_type**: Code for the type of room assigned to the booking. Sometimes the assigned room type differs from the reserved room type due to hotel operation reasons (e.g. overbooking) or by customer request. Code is presented instead of designation for anonymity reasons
- **booking_changes**: Number of changes made to the booking
- **deposit_type**: Type of deposit made: No Deposit – no deposit was made; Non Refund – a deposit was made in the value of the total stay cost; Refundable – a deposit was made with a value under the total cost of stay

#### Agent & Company

- **agent**: ID of the travel agent who made the booking
- **company**: ID of the company that made the booking (if applicable)
- **days_in_waiting_list**: Number of days the booking was on the waiting list

#### Customer & Pricing

- **customer_type**: Type of customer: Transient, Contract, Group, or Transient-party
- **adr**: Average Daily Rate — calculated as total revenue divided by total nights stayed
- **required_car_parking_spaces**: Number of car parking spaces requested
- **total_of_special_requests**: Total number of special requests (e.g., sea view, high floor, twin bed, etc.)

#### Reservation Status

- **reservation_status**: Final status of the reservation: Check-Out, Canceled, or No-Show
- **reservation_status_date**: Date on which the last status was recorded

#### 1.1 Explore

In [4]:
df = pd.read_csv(file_dst)

pd.set_option('display.max_columns', None)
print("\n Dataset Overview:")
df.head() 
#print(f"\n Shape: {df.shape}")
#print(f" Columns: {df.columns.tolist()}")
#print(f"\n Missing Values:\n{df.isnull().sum().sort_values(ascending=False).head(10)}")


 Dataset Overview:


,hotel,is_canceled,lead_time,arrival_date_year,arrival_date_month,arrival_date_week_number,arrival_date_day_of_month,stays_in_weekend_nights,stays_in_week_nights,adults,children,babies,meal,country,market_segment,distribution_channel,is_repeated_guest,previous_cancellations,previous_bookings_not_canceled,reserved_room_type,assigned_room_type,booking_changes,deposit_type,agent,company,days_in_waiting_list,customer_type,adr,required_car_parking_spaces,total_of_special_requests,reservation_status,reservation_status_date
0,Resort Hotel,0,342,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,3,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
1,Resort Hotel,0,737,2015,July,27,1,0,0,2,0.0,0,BB,PRT,Direct,Direct,0,0,0,C,C,4,No Deposit,NaN,NaN,0,Transient,0.0,0,0,Check-Out,2015-07-01
2,Resort Hotel,0,7,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Direct,Direct,0,0,0,A,C,0,No Deposit,NaN,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
3,Resort Hotel,0,13,2015,July,27,1,0,1,1,0.0,0,BB,GBR,Corporate,Corporate,0,0,0,A,A,0,No Deposit,304.0,NaN,0,Transient,75.0,0,0,Check-Out,2015-07-02
4,Resort Hotel,0,14,2015,July,27,1,0,2,2,0.0,0,BB,GBR,Online TA,TA/TO,0,0,0,A,A,0,No Deposit,240.0,NaN,0,Transient,98.0,0,1,Check-Out,2015-07-03


In [5]:
df.shape

(119390, 32)

#### 1.2 Missing Values
- Large missing value **company (94%)** we should drop becuse it doesn't have much impact.
- Used median imputation for numerical fields **agent (13.6%)**, **children(0.003%)** and mode for categorical **country (0.4%)**) 

In [6]:
null = pd.DataFrame({'Null Values' : df.isna().sum(), 'Percentage Null Values' : (df.isna().sum()) / (df.shape[0]) * (100)})
null.sort_values(by='Null Values', ascending=False)

,Null Values,Percentage Null Values
company,112593,94.306893
agent,16340,13.686238
country,488,0.408744
children,4,0.003350
reserved_room_type,0,0.000000
assigned_room_type,0,0.000000
booking_changes,0,0.000000
deposit_type,0,0.000000
hotel,0,0.000000
previous_cancellations,0,0.000000


In [7]:
df = df.drop(['company'], axis=1)
df.fillna({
    'agent': df['agent'].median(),
    'country': df['country'].mode()[0],
    'children': df['children'].mode()[0]
}, inplace=True)


#### 1.2 Data type conversion
- children from float64 to int64
- agent from float64 to int64
- reservation_status_data from object to datetime64

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 119390 entries, 0 to 119389
Data columns (total 31 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   hotel                           119390 non-null  object 
 1   is_canceled                     119390 non-null  int64  
 2   lead_time                       119390 non-null  int64  
 3   arrival_date_year               119390 non-null  int64  
 4   arrival_date_month              119390 non-null  object 
 5   arrival_date_week_number        119390 non-null  int64  
 6   arrival_date_day_of_month       119390 non-null  int64  
 7   stays_in_weekend_nights         119390 non-null  int64  
 8   stays_in_week_nights            119390 non-null  int64  
 9   adults                          119390 non-null  int64  
 10  children                        119390 non-null  float64
 11  babies                          119390 non-null  int64  
 12  meal            

In [9]:
df['children'] = df['children'].fillna(0).astype(int)
df['agent'] = df['agent'].astype('Int64')
df['reservation_status_date'] = pd.to_datetime(df['reservation_status_date'], errors='coerce')
print(df.dtypes[['children', 'agent', 'reservation_status_date']])


children                            int64
agent                               Int64
reservation_status_date    datetime64[ns]
dtype: object


#### 1.3 Outliers
- Remove outliers data:
  - **1) Over and Under pricing**, adr under pricing (-6.38), over pricing (5,000) by remove and IQR method
  - **2) Booking without no guests**, adults < 0
  - **3) Limit Stay Duration**, stays_in_week_nights should booking not more than 30 days.
  - **4) Cap Waiting Days**, days_in_waiting_list shoud not more than 180 days.

In [10]:
df.describe().T


,count,mean,min,25%,50%,75%,max,std
is_canceled,119390.0,0.370416,0.0,0.0,0.0,1.0,1.0,0.482918
lead_time,119390.0,104.011416,0.0,18.0,69.0,160.0,737.0,106.863097
arrival_date_year,119390.0,2016.156554,2015.0,2016.0,2016.0,2017.0,2017.0,0.707476
arrival_date_week_number,119390.0,27.165173,1.0,16.0,28.0,38.0,53.0,13.605138
arrival_date_day_of_month,119390.0,15.798241,1.0,8.0,16.0,23.0,31.0,8.780829
stays_in_weekend_nights,119390.0,0.927599,0.0,0.0,1.0,2.0,19.0,0.998613
stays_in_week_nights,119390.0,2.500302,0.0,1.0,2.0,3.0,50.0,1.908286
adults,119390.0,1.856403,0.0,2.0,2.0,2.0,55.0,0.579261
children,119390.0,0.103886,0.0,0.0,0.0,0.0,10.0,0.398555
babies,119390.0,0.007949,0.0,0.0,0.0,0.0,10.0,0.097436


In [11]:
# 1) Remove over and under pricing
Q1 = df['adr'].quantile(0.25)
Q3 = df['adr'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Remove negative and too high prices
df = df[(df['adr'] > 0) & (df['adr'] < 5000)] 
# Remove IQR outliers
df = df[(df['adr'] >= lower_bound) & (df['adr'] <= upper_bound)] 


# 2) Booking without no guests
df = df[df['adults'] >= 1]

# 3) Limit Stay Duration
df['total_nights'] = df['stays_in_week_nights'] + df['stays_in_weekend_nights']
df = df[df['total_nights'] <= 30]

# 4) Cap Waiting Days
df['days_in_waiting_list'] = df['days_in_waiting_list'].clip(upper=180)


In [17]:
df.describe().T

,count,mean,min,25%,50%,75%,max,std
is_canceled,113370.0,0.374482,0.0,0.0,0.0,1.0,1.0,0.483991
lead_time,113370.0,106.113363,0.0,19.0,71.0,164.0,709.0,107.775639
arrival_date_year,113370.0,2016.148179,2015.0,2016.0,2016.0,2017.0,2017.0,0.706403
arrival_date_week_number,113370.0,27.016433,1.0,16.0,27.0,38.0,53.0,13.719448
arrival_date_day_of_month,113370.0,15.772506,1.0,8.0,16.0,23.0,31.0,8.786167
stays_in_weekend_nights,113370.0,0.927441,0.0,0.0,1.0,2.0,10.0,0.982623
stays_in_week_nights,113370.0,2.497169,0.0,1.0,2.0,3.0,22.0,1.851449
adults,113370.0,1.854291,1.0,2.0,2.0,2.0,4.0,0.47132
children,113370.0,0.079263,0.0,0.0,0.0,0.0,10.0,0.340159
babies,113370.0,0.007392,0.0,0.0,0.0,0.0,10.0,0.094751


In [18]:
df.shape

(113370, 32)